In [37]:
#import open AI 
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.indexes import VectorstoreIndexCreator
from detectron2.config import get_cfg

#import other 
import numpy as np 
import pandas as pd 
import os 
import re 
import time

Open AI API Key is necessary here as the vectorized database will be queried by OpenAPI (must be ways to go around this...)

In [38]:
os.environ['OPENAI_API_KEY'] = "sk-proj-Z3azBQxgMb14LYqqWJeUT3BlbkFJXe04TidhcC9bElNpy5P5"
cfg = get_cfg()    
cfg.MODEL.DEVICE = 'cpu' #GPU is recommended

This is where the magic happens

The .pdf file is loaded (1) and then transformed into a Vector Database (2) 

In [12]:
#Load the list of all pdf files 
pdf_directory_list =  [filename for filename in os.listdir("./") if (('.pdf' in filename) and ('Hadleigh' not in filename))]
pdf_directory_list

['ANZ_climate-report-2023.pdf',
 'Sustainability Review & Climate Related Disclosures - FY23.pdf',
 'RLAL-NZ Climate-Related Disclosures 2023.pdf',
 'SRLHA NZ CRD Statement_signed.pdf',
 'ICBC Ltd CRD 31 Dec 23  Final - For submission.pdf',
 'HSBC NZ 2023 Climate Statement lodgement letter - signed.pdf',
 'RBD 2023 CRD.pdf',
 '2023 Climate Statements for Citibank N.A. (New Zealand Branch).pdf',
 'Z Climate Statements 2023 - FINAL.pdf',
 'Ventia - 2023 Sustainability Report.pdf',
 'SC9600 Climate Related Disclosure Final Copy.pdf',
 'QBE_NZ Climate Statement.pdf',
 'Rabobank New Zealand Banking Group 2023 Climate Statements.pdf',
 'AIA NZ Climate Related Disclosers Report FINAL_signed.pdf',
 'Resilient infrastructure for a decarbonising world (7) - FINAL 28.02.24 9.30pm.pdf',
 'NZL Climate Related Disclosure.pdf',
 'MIL0019_Climate_Related_Disclosures_2023.pdf',
 'CDL0017_Climate_Related_Disclosures_2023.pdf',
 'asb-climate-report-2022.pdf',
 'CRD 2023 CEO Declaration.pdf',
 'Citibank N

## PART 1 - EXAMPLE 

In [24]:
filename = pdf_directory_list[0]
loader = [UnstructuredPDFLoader(f"./{filename}")]
index = VectorstoreIndexCreator().from_loaders(loader)

/opt/conda/lib/python3.10/site-packages/pdfminer/pdfpage.py:159: PDFTextExtractionNotAllowedWarning: The PDF <_io.BufferedReader name='./ANZ_climate-report-2023.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding.
  warnings.warn(warning_msg, PDFTextExtractionNotAllowedWarning)


Example of a Query

In [7]:
query = "List of climate change scenarios in this document?"
index.query(query)

' Net Zero 2050 (Orderly Category), Delayed Transition (Disorderly Category), Current Policies (Hot House World Category)'

In [8]:
query = "Who are the providers of climate change scenarios used in this document?"
index.query(query)

' The Network for Greening the Financial System (NGFS) scenarios were used as the basis for the sector-level scenarios.'

In [11]:
query = "Do they use scenarios developed by their respective industry/sector, if yes, what is the name?"
index.query(query)

' The scenarios used by Briscoe Group were developed by the Network for Greening the Financial System (NGFS).'

In [13]:
query = "Do they mix multiple scenarios from multiple data sources? If yes, which sources"
index.query(query)

' Yes, the Briscoe Group used a combination of scenarios from multiple data sources, including external experts and industry participants, to assess the potential risks and opportunities of climate change on their business. These sources included the Network for Greening the Financial System (NGFS) scenarios, as well as their own internal risk assessment framework.'

In [28]:
query = "Do they mix multiple scenarios from multiple data sources? If yes, which sources"
index.query(query)

' Yes, ANZ uses multiple scenarios from multiple data sources, including their own ANZ-specific climate scenario narratives and publicly available source scenarios and data.'

This is weird !!!!! We should use normal NLP analysis then! 

In [19]:
query = "Word counts in this document"
index.query(query)

" I don't know."

In [17]:
long_template = """
You're a researcher analysing climate disclosure statements to see how climate change is framed. 
You are assessing each statement for key information and writing an analysis in a specific format. 
Only give the analysis, not the statement, following the exact format provided below:
------------------------
ANALYSIS:
Number of words in this document: 20
Number of negative words: 20
Number of positive words: 50
Seperate climate statement or combine with annual report: Seperate 
Company: Briscoes
List of climate change scenarios: ["Net Zero 2050 (Orderly)", "Delayed Transition (Disorderly Category)", "Current Policies (Hot House World Category)"]
Equivalent global temperature outcome due to the scenarios by 2100: ["1.4oC","1.6oC","3oC"]
Scenario provider: ["Network for Greening the Financial System (NGFS)","Climate Change Commission"]
Do they use industry-specific scenarios, if yes, which one: ["Integrated climate change scenarios for NZ retail sector"]
Do they mix multiple scenarios from multiple data sources: "Yes, mix and pick with SSPs, CCCs"
Do they use baseline scenarios: "No"
Non-temperature variables covered in scenarios: [carbon price,  oil price volatility ,  technology change , customer attitude , acro economic factors, access to finance and insurance] 
What are the timeframe: [Intermmediate: 3 year (2025), Short-term: 10 years, Median term: 30 years, Longterm : 50 years ]
Physical hazards relevant to the busines: [Sea-level rise,   Precipitation, Drought ]
Is discussion of physical risk based on the geo-locations of its assets: "Yes, they have breakdown of factories"
Is there quantified risk estimate: "Yes, 50% assets are exposed to SLR"
----------------------------------
Now do the same analysis for the document. Follow the same analysis format from above exactly. If you can't find an answer, use N/A:
-----------------------------------
"""
print(index.query(long_template))


ANALYSIS:
Number of words: 20
Number of negative words: 20
Number of positive words: 50
Seperate climate statement or combine with annual report: Seperate 
Company: Briscoes
List of climate change scenarios: ["Net Zero 2050 (Orderly)", "Delayed Transition (Disorderly Category)", "Current Policies (Hot House World Category)"]
Equivalent global temperature outcome due to the scenarios by 2100: ["1.4oC","1.6oC","3oC"]
Scenario provider: ["Network for Greening the Financial System (NGFS)","Climate Change Commission"]
Do they use industry-specific scenarios, if yes, which one: ["Integrated climate change scenarios for NZ retail sector"]
Do they mix multiple scenarios from multiple data sources: "Yes, mix and pick with SSPs, CCCs"
Do they use baseline scenarios: "No"
Non-temperature variables covered in scenarios: [carbon price,  oil price volatility ,  technology change , customer attitude , acro economic factors, access to finance and insurance] 
What are the timeframe: [Intermmediate: 3 

## PART 2- ACTUAL ANALYSIS

We would have two options in here:
- We ask them question by question => so we reduce the number of tokens on the questions and have more tokens available for answer
- We ask them in a template prepare beforehand => so we have more structured dataset

In [4]:
#generate a dictionary of question 
query_dict = {
'q1': 'What is the Word counts/ Page counts/ Length of document?',
'q2': 'Is the climate risk disclosure issued as a separate climate statement, sustainablity report, or annual report?',
'q3': 'List the adoption provisions that this statement applies. If no, plaese answer NA',
'q4': 'Provide the list of all climate scenarios or pathways that have been used in this statement', 
'q5': 'Provide the global temperature outcomes associated with the climate scenarios by 2100',
'q6': 'Provide the list of scenario providers used in this statement',
'q7': 'Are the scenarios included a pick and mix of different scenario providers, such as SSPs or RCPs?', 
'q8': 'Are the scenarios off-the-shelf or are they further customized to meet with company business models?',
'q9': 'What are the main variables that are covered and drive the scenarios? (apart from global temperature outcomes)?',
'q10': 'Are the scenarios/assessments being co-produced ?  For instance, are there keywords like ‘participatory’, ‘co-creat$’, ‘deliber$’, ‘co-design$’ which might be indicative of the legitimacy of the scenarios?',
'q11': 'Is there disclosure of the  baseline or reference scenarios or assumptions that is used to measure the impact of climate change scenarios against?',
'q12': 'What climate models/ integrated assessment models / macro-economic models used in the creation of scenarios?',
'q13': 'Is damage from extreme or acute climate events considered in the set of climate scenarios/ pathways?',
'q14': 'Are the scenarios based on global climate models or local, New Zealand based models or regional models?',
'q15': 'What are the short, medium and long term timeframes and their associated rationale?',
'q16': 'Does the statement provide justification for their choice of climate scenarios/ models?',
'q17': 'What are the physical hazards listed in the climate statement that have an impact on firms?',
'q18': 'Is the discussion of physical risk generic or based on the geo-locations of its assets?',
'q19': 'What are the business risks specific to the firm due to physical hazards?',
'q20': 'What are the opportunities specific to the business due to physical hazards?',
'q21': 'Is there any quantification of the business exposure to physical hazards across climate scenarios ? What are they?', 
'q22': 'Are the current financial impacts of climate change discussed in the disclosure? What are they?',
'q23': 'Are the future financial impacts of climate change discussed in the disclosure? What are they?', 
'q24': 'Is there disclosure of the range of uncertainty surrounding the expected (average - mean) exposure and financial impacts due to climate physical hazards?',
'q25': 'Are the financial impacts detailed  quantatively or qualitatively?',
'q26': 'Are the disclosure of financial impact focus on direct or indirect loss?',
'q27': 'Did science agencies  involve in the creation of this disclosure? What are they? (for instance, NIWA)?',
'q28': 'Did science agencies involve in terms of providing the climate scenarios/consulting on the identification of climate risk and opportunities/ analysis of climate physical risk, hazard, exposure/ providing datasets or analysis of current/ anticipated impacts?',
'q29': 'Was a consultancy employed to conduct part, or all, of the disclosure?',
'q30': 'What parts of the disclosures (climate risk/ scenario creation/ data provision / transition risk) are consultants commonly contracted',
'q31': 'Is the overall discussion about climate risk neutral/ negative / positve ?',
'q32': "Is climate risk expected to have a significant impact on the company's financial performance?",
'q33': 'Is the overall discussion in the document qualitative / quantitative (%) / financially quantitative ($)?'
}


In [5]:
pdf_directory_list

['ANZ_climate-report-2023.pdf',
 'Sustainability Review & Climate Related Disclosures - FY23.pdf',
 'RLAL-NZ Climate-Related Disclosures 2023.pdf',
 'SRLHA NZ CRD Statement_signed.pdf',
 'ICBC Ltd CRD 31 Dec 23  Final - For submission.pdf',
 'HSBC NZ 2023 Climate Statement lodgement letter - signed.pdf',
 'RBD 2023 CRD.pdf',
 '2023 Climate Statements for Citibank N.A. (New Zealand Branch).pdf',
 'Z Climate Statements 2023 - FINAL.pdf',
 'Ventia - 2023 Sustainability Report.pdf',
 'SC9600 Climate Related Disclosure Final Copy.pdf',
 'QBE_NZ Climate Statement.pdf',
 'Rabobank New Zealand Banking Group 2023 Climate Statements.pdf',
 'AIA NZ Climate Related Disclosers Report FINAL_signed.pdf',
 'Resilient infrastructure for a decarbonising world (7) - FINAL 28.02.24 9.30pm.pdf',
 'NZL Climate Related Disclosure.pdf',
 'MIL0019_Climate_Related_Disclosures_2023.pdf',
 'CDL0017_Climate_Related_Disclosures_2023.pdf',
 'asb-climate-report-2022.pdf',
 'CRD 2023 CEO Declaration.pdf',
 'Citibank N

In [40]:
################
#TEST CASE     #
################
# Let test other one => Let goes for Chubb insurance because they are the most comperehensive ones I feel  
filename = 'Chubb Life Insurance New Zealand Limited Climate Disclosure 2023.pdf'
loader = [UnstructuredPDFLoader(f"./{filename}")]
index = VectorstoreIndexCreator().from_loaders(loader)

# Create a data dictionary to put the result into the analysis 
answers = dict()
for question_number, question in query_dict.items():
    print(question_number, question)
    answer = index.query(question)
    print(answer)
    answers[question_number] = answer
    answers['filename'] = filename
    # Sleep for 5 seconds to avoid hitting OpenAI Rate Limits
    time.sleep(2)

1 What is the Word counts/ Page counts/ Length of document?
 I don't know.
2 Is the climate risk disclosure issued as a separate climate statement, sustainablity report, or annual report?
 The climate risk disclosure is issued as a separate climate statement.
3 List the adoption provisions that this statement applies. If no, plaese answer NA
 Adoption provisions 1, 2, 3, 4, 5, 6, 7
4 Provide the list of all climate scenarios or pathways that have been used in this statement
 The scenarios used in this statement are the Orderly, Too Little Too Late, and Hothouse scenarios, as well as the 1.5° C and >2° C pathways.
5 Provide the global temperature outcomes associated with the climate scenarios by 2100


The global temperature outcomes associated with the climate scenarios by 2100 are as follows:

- Orderly scenario: +1.5°C
- Too Little Too Late scenario: >2°C
- Hothouse scenario: >3°C
6 Provide the list of scenario providers used in this statement
 The Orderly and Hothouse scenarios are 

In [42]:
answers_dataframe = pd.DataFrame([answers])
answers_dataframe

,1,filename,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,I don't know.,Chubb Life Insurance New Zealand Limited Clima...,The climate risk disclosure is issued as a se...,"Adoption provisions 1, 2, 3, 4, 5, 6, 7",The scenarios used in this statement are the ...,\n\nThe global temperature outcomes associated...,The Orderly and Hothouse scenarios are common...,"No, the scenarios included in this analysis a...",The scenarios are further customized to meet ...,The main variables that are covered and drive...,...,"No, there is no quantification of the busines...","No, the current financial impacts of climate ...","Yes, the future financial impacts of climate ...",Yes,The financial impacts are detailed qualitativ...,The disclosure of financial impact focuses on...,"Yes, science agencies were involved in the cr...",It is not explicitly stated in the context pr...,"\nBased on the context provided, it does not a...",Consultants are commonly contracted to assist...


In [15]:
pdf_directory_list[-8:]

['nzm-cs-dec-2023.pdf',
 'CCBNZBG Climate Report for the year ended 31 December 2023.pdf',
 'BNZ_2023-Climate-Report.pdf',
 'Seeka Limited Climate Disclosure 2023.pdf',
 'kiwibank_sustainability_report2023.pdf',
 'Climate-Report-2023-Westpac-NZ.pdf',
 'NZME Annual Report 2023 (including climate statement).pdf',
 'ICBC NZ Ltd CRD 31 Dec 2023.pdf']

In [16]:
#############################################################################
#FULL IMPLEMENTATION                                                        #
#PLEASE ONLY DO THIS IF YOU ARE SURE ABOUT YOUR CODE - AS IT MAY COST MONEY # 
#############################################################################
full_answers = pd.DataFrame()
for filename in pdf_directory_list:
    print(filename)
    try:
        loader = [UnstructuredPDFLoader(f"./{filename}")]
        index = VectorstoreIndexCreator().from_loaders(loader)
        # Create a data dictionary to put the result into the analysis 
        answers = dict()
        for question_number, question in query_dict.items():
            print(question_number, question)
            answer = index.query(question)
            answers[question_number] = answer
            answers['filename'] = filename      
            # Sleep for 5 seconds to avoid hitting OpenAI Rate Limits
            time.sleep(2)
        # Convert the dictionary to a DataFrame and specify it should be a single row
        row_df = pd.DataFrame([answers])  # Note the [row] to convert the single dictionary to a DataFrame
        full_answers = pd.concat([full_answers, row_df], ignore_index=True)
    except:
        print('Error in reading the file')


columns = ['filename'] + [str(column) for column in full_answers.columns if 'filename' not in str(column)]
full_answers[columns].head(5)
full_answers[columns].to_csv('full_answers.csv')

nzm-cs-dec-2023.pdf
q1 What is the Word counts/ Page counts/ Length of document?
q2 Is the climate risk disclosure issued as a separate climate statement, sustainablity report, or annual report?
q3 List the adoption provisions that this statement applies. If no, plaese answer NA
q4 Provide the list of all climate scenarios or pathways that have been used in this statement
q5 Provide the global temperature outcomes associated with the climate scenarios by 2100
q6 Provide the list of scenario providers used in this statement
q7 Are the scenarios included a pick and mix of different scenario providers, such as SSPs or RCPs?
q8 Are the scenarios off-the-shelf or are they further customized to meet with company business models?
q9 What are the main variables that are covered and drive the scenarios? (apart from global temperature outcomes)?
q10 Are the scenarios/assessments being co-produced ?  For instance, are there keywords like ‘participatory’, ‘co-creat$’, ‘deliber$’, ‘co-design$’ whic

/opt/conda/lib/python3.10/site-packages/pdfminer/pdfpage.py:159: PDFTextExtractionNotAllowedWarning: The PDF <_io.BufferedReader name='./CCBNZBG Climate Report for the year ended 31 December 2023.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding.
  warnings.warn(warning_msg, PDFTextExtractionNotAllowedWarning)


q1 What is the Word counts/ Page counts/ Length of document?
q2 Is the climate risk disclosure issued as a separate climate statement, sustainablity report, or annual report?
q3 List the adoption provisions that this statement applies. If no, plaese answer NA
q4 Provide the list of all climate scenarios or pathways that have been used in this statement
q5 Provide the global temperature outcomes associated with the climate scenarios by 2100
q6 Provide the list of scenario providers used in this statement
q7 Are the scenarios included a pick and mix of different scenario providers, such as SSPs or RCPs?
q8 Are the scenarios off-the-shelf or are they further customized to meet with company business models?
q9 What are the main variables that are covered and drive the scenarios? (apart from global temperature outcomes)?
q10 Are the scenarios/assessments being co-produced ?  For instance, are there keywords like ‘participatory’, ‘co-creat$’, ‘deliber$’, ‘co-design$’ which might be indicativ

In [23]:
#Combine result from multiple runs 
full_answers = pd.read_csv('./full_answers.csv')
full_answers_additional = pd.read_csv('./full_answers_additional.csv')
full_answers = pd.concat([full_answers, full_answers_additional], ignore_index = True)
# Renaming columns using the dictionary
full_answers.rename(columns=query_dict, inplace=True)
# Strip line break
# Function to remove leading line breaks
def remove_leading_line_breaks(text):
    if isinstance(text, str):
        return text.lstrip('\n')
    return text

# Apply the function to the entire DataFrame
full_answers = full_answers.applymap(remove_leading_line_breaks)

full_answers.to_csv('./full_answers.final.csv')
full_answers.head(5)

/tmp/ipykernel_4784/3997302952.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  full_answers = full_answers.applymap(remove_leading_line_breaks)


,Unnamed: 0,filename,What is the Word counts/ Page counts/ Length of document?,"Is the climate risk disclosure issued as a separate climate statement, sustainablity report, or annual report?","List the adoption provisions that this statement applies. If no, plaese answer NA",Provide the list of all climate scenarios or pathways that have been used in this statement,Provide the global temperature outcomes associated with the climate scenarios by 2100,Provide the list of scenario providers used in this statement,"Are the scenarios included a pick and mix of different scenario providers, such as SSPs or RCPs?",Are the scenarios off-the-shelf or are they further customized to meet with company business models?,...,Is there disclosure of the range of uncertainty surrounding the expected (average - mean) exposure and financial impacts due to climate physical hazards?,Are the financial impacts detailed quantatively or qualitatively?,Are the disclosure of financial impact focus on direct or indirect loss?,"Did science agencies involve in the creation of this disclosure? What are they? (for instance, NIWA)?","Did science agencies involve in terms of providing the climate scenarios/consulting on the identification of climate risk and opportunities/ analysis of climate physical risk, hazard, exposure/ providing datasets or analysis of current/ anticipated impacts?","Was a consultancy employed to conduct part, or all, of the disclosure?",What parts of the disclosures (climate risk/ scenario creation/ data provision / transition risk) are consultants commonly contracted,Is the overall discussion about climate risk neutral/ negative / positve ?,Is climate risk expected to have a significant impact on the company's financial performance?,Is the overall discussion in the document qualitative / quantitative (%) / financially quantitative ($)?
0,0,ANZ_climate-report-2023.pdf,I don't know the exact word count or page cou...,The climate risk disclosure is issued as a se...,This statement applies to the adoption provis...,1. 'Orderly' 1.50C\n2. Shared Socioeconomic Pa...,The global temperature outcomes associated wit...,The Intergovernmental Panel on Climate Change...,"Yes, the scenarios included a pick and mix of...",The scenarios used by ANZ are a combination o...,...,"Yes, there is disclosure of the range of unce...",The financial impacts are detailed qualitativ...,The disclosure of financial impact focuses on...,"Yes, NIWA was involved in the creation of thi...","Yes, science agencies were involved in provid...",I don't know.,Consultants are commonly contracted for scena...,Neutral,"Yes, climate risk is expected to have a signi...",The overall discussion in the document is qua...
1,1,Sustainability Review & Climate Related Disclo...,I don't have enough information to accurately...,The climate risk disclosure is issued as part ...,Adoption Provision 2,The statement mentions three climate scenario...,The 'orderly' 1.5ºC scenario results in a glo...,The scenario providers used in this statement...,"No, the scenarios were developed using the si...",The scenarios were customized to meet Summers...,...,Yes,The financial impacts are detailed qualitativ...,The disclosure of financial impact focuses on...,"Yes, NIWA (National Institute of Water and At...","Yes, science agencies were involved in provid...",Yes\n\nYes,Consultants are commonly contracted for scena...,The overall discussion about climate risk is ...,"Based on the information provided, it is not p...",The overall discussion in the document is qua...
2,2,RLAL-NZ Climate-Related Disclosures 2023.pdf,The document is 12 pages long.,The climate risk disclosure is issued as a se...,Adoption Provision 1: Current financial impact...,The climate scenarios or pathways used in thi...,Orderly: 1.5°C\nToo Little Too Late: >2°C\nHot...,"OECD, IEA, IPCC, NGFS, NIWA, CCC","Yes, the scenarios included in the assessment...",The scenarios used are further customized to ...,...,No,The financial impacts are det